In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy.signal import butter, filtfilt,welch
from sklearn.preprocessing import MinMaxScaler
import os
from scipy.stats import gaussian_kde
import seaborn as sns

In [2]:
data=pd.read_csv(r'D:\\1.5inch Pump Data\\With Vibration Sensor Test\\02_P1_20_100-50_Cracked Diaphragm.csv')

In [ ]:
data.columns

In [4]:
vib_data = pd.DataFrame(data,columns=['P1-Air-Supply-pressure','P1-CPM','Right-Vibration-Data'])

In [5]:
vib_data.rename(columns={'Right-Vibration-Data':'Left-Vibration-Data'},inplace=True)

In [ ]:
vib_data.info()

In [7]:
#outlier analysis
q1 = vib_data['Left-Vibration-Data'].quantile(0.25)
q3 = vib_data['Left-Vibration-Data'].quantile(0.75)
IQR = q3 - q1
upper_limit = q3 + (1.5 * IQR)   
lower_limit = q1 - (1.5 * IQR)
clean_vibration_dat = vib_data[(vib_data['Left-Vibration-Data']<upper_limit)&(vib_data['Left-Vibration-Data']>lower_limit)]
clean_vibration_dat.reset_index(drop=True,inplace=True)

In [48]:
def df_cleaning(datframe):
    print("Infinity values\n",np.isinf(datframe).sum())
    print("NaN values\n",datframe.isna().sum())
    print("Null vlaues\n",datframe.isnull().sum())
    return df_cleaning

In [ ]:
df_cleaning(clean_vibration_dat)

In [ ]:
len(clean_vibration_dat)

In [ ]:
clean_vibration_dat.head()

In [ ]:
#Create timestamp column with sampling frequency 
clean_vibration_dat['Timestamp'] = pd.date_range('2024-07-13 11:41:46', freq='30ms', periods=len(clean_vibration_dat))
#Convert timestamp data type object in to datetime
clean_vibration_dat['Timestamp'] =pd.to_datetime(clean_vibration_dat['Timestamp'])
#Set timestamp as index
clean_vibration_dat.set_index(['Timestamp'], inplace=True)

In [ ]:
plt.figure(figsize=(20,10))
plt.subplot(211)
plt.plot(clean_vibration_dat.index, clean_vibration_dat['P1-CPM'],'r')
plt.legend(['P1-CPM'])

plt.subplot(212)
#plt.plot(cylinder_1_df.index, cylinder_1_df['PresTrans1'])
plt.plot(clean_vibration_dat.index, clean_vibration_dat['Left-Vibration-Data'])
plt.legend(['Left-Vibration-Data'])
plt.tight_layout()

In [ ]:
clean_vibration_dat.head()

In [9]:
# Step 2: Define the high-pass filter
def high_pass_filter(data, cutoff, fs, order=5):
    nyquist = 0.5 * fs
    normal_cutoff = cutoff / nyquist
    b, a = butter(order,normal_cutoff, btype='high', analog=False)
    y = filtfilt(b, a, data)
    return y
# Parameters for the high-pass filter
cutoff_frequency = 0.5  # Hz 
sampling_rate = 33.3  # Hz (adjust according to your actual sampling rate)
filtered_data = high_pass_filter(clean_vibration_dat.iloc[:,2], cutoff_frequency, sampling_rate)

In [10]:
Filter_data_df = pd.DataFrame(filtered_data,columns=['Left-Vibration-Data'])

In [11]:

Filter_data_df['newtime'] = pd.date_range('2024-04-11 00:00:00.000000', freq='30ms', periods=len(Filter_data_df))

Filter_data_df['newtime'] =pd.to_datetime(Filter_data_df['newtime'])

Filter_data_df.set_index(['newtime'], inplace=True)

In [ ]:
plt.figure(figsize=(30,5))
#plt.plot(cylinder_1_df.index, cylinder_1_df['PresTrans1'])
plt.plot(Filter_data_df.index, Filter_data_df['Left-Vibration-Data'])
plt.legend(['Good Diaphragm Vibration Data'])

In [ ]:
len(Filter_data_df)

In [ ]:
total_hrs=0
hrs=np.round((((Filter_data_df.index[-1]-Filter_data_df.index[0]).total_seconds())/3600),decimals=1)
total_hrs=total_hrs+hrs
total_hrs

In [ ]:
Filter_data_df.info()

In [21]:
vibration_array_data=Filter_data_df['Left-Vibration-Data'].values

In [ ]:
Filter_data_df['Right-Vibration-Data'].values
x_freq2,y_amplitude2=welch(vibration_array_data,fs=30)
plt.plot(x_freq2,y_amplitude2)

# FFT Analysis

In [61]:
# Function to apply a high-pass filter
def high_pass_filter(data, cutoff, fs, order=5):
    nyquist = 0.5 * fs
    normal_cutoff = cutoff / nyquist
    b, a = butter(order, normal_cutoff, btype='high', analog=False)
    filtered_data = filtfilt(b, a, data)
    return filtered_data

In [62]:

#temp_dic = {}
Freq_list = [] 
Amp_list = []
Freq_Amp_df = pd.DataFrame(columns=['Freq','Amp'])

start_time = Filter_data_df.index[0]
end_time = Filter_data_df.index[-1]
time_interval = pd.Timedelta(minutes=2)

current_time = start_time
while current_time <= end_time:
    
    next_time = current_time + time_interval
    
    # Slice the DataFrame based on the specified time range
    sliced_df = Filter_data_df.iloc[(Filter_data_df.index >= current_time) & (Filter_data_df.index <= next_time)]
    
    # Perform FFT analysis only if there is data in the sliced DataFrame
    n = len(sliced_df)
    if n > 0:
        t = sliced_df.index
        s = sliced_df['Left-Vibration-Data']

        # Apply high-pass filter
        dt = (t[1] - t[0]).total_seconds()  # Sampling interval in seconds
        

        # FFT
        s -= s.mean()
        fr = np.fft.rfftfreq(n, dt)
        fou = np.fft.rfft(s)
        
        # Create a DataFrame for the current fr,amp
        temp_df = pd.DataFrame({'Frequency': fr,'Amplitude': np.abs(fou)})

        #Filter max apllitude frim temp_df and put it in to df_filter
        df_Filter = temp_df.loc[temp_df['Amplitude'].idxmax()]
   
        #Append coresponding maximum amplitude frequency values in to freq_amp list
        Freq_list.append((df_Filter['Frequency']))
        
        Amp_list.append((df_Filter['Amplitude']))
                 
    else:
        pass
    current_time = next_time    
#temp_dic[key] = result_df

# Append the DataFrame to the list
Freq_Amp_df = pd.DataFrame({'Freq':Freq_list,'Amp':Amp_list})

In [ ]:
Freq_Amp_df=Freq_Amp_df[Freq_Amp_df['Freq']>0.03]

In [63]:
#Reset the data frame
Freq_Amp_df.reset_index(drop=True,inplace=True)

#Normalize the data frame values
normalized_variables = MinMaxScaler().fit_transform(Freq_Amp_df)

#Insert the normalized vaules in to the new columns
for value_index,col_name in enumerate(Freq_Amp_df.columns):
    Freq_Amp_df["Norm_{}".format(col_name)] = normalized_variables[:,value_index]

#Create Diaprhagm liftime
timeline = [n for n in np.linspace(0,total_hrs,len(Freq_Amp_df))]

#Insert timeline in to dataframe
Freq_Amp_df['Diaphragm_life'] = timeline

#Set Diaphragm life as index
Freq_Amp_df.set_index(['Diaphragm_life'],inplace=True)

In [64]:
def plot_funtion(dataframe):
    a = len(dataframe.columns)  # number of rows
    b = 1  # number of columns
    c = 1  # initialize plot counter
    fig1 = plt.figure(figsize=(35,20))
       
    for i in dataframe.columns:
        # Create a subplot and save the plot
        plt.subplot(a, b, c)
        plt.plot(dataframe.index, dataframe[i],marker = '.')
        plt.xticks(np.arange(np.min(dataframe.index),np.max((dataframe.index)),0.2))             
        #plt.ylim(0,0.5)
        plt.title(f"{i}{' vs Diaphragm Life Time'}")
        #plt.legend(df.columns, loc='lower center', bbox_to_anchor=(0.5, -0.14), ncol=8)
        c = c + 1
    return plot_funtion

In [ ]:
plot_funtion(Freq_Amp_df)

# FFT Spectrum Plot

In [42]:

#temp_dic = {}
Freq_list = [] 
Amp_list = []
Freq_Amp_df = pd.DataFrame(columns=['Freq','Amp'])

start_time = Filter_data_df.index[0]
end_time = Filter_data_df.index[-1]
time_interval = pd.Timedelta(minutes=1)

current_time = start_time
while current_time <= end_time:
    
    next_time = current_time + time_interval
    
    # Slice the DataFrame based on the specified time range
    sliced_df = Filter_data_df.iloc[(Filter_data_df.index >= current_time) & (Filter_data_df.index <= next_time)]
    
    # Perform FFT analysis only if there is data in the sliced DataFrame
    n = len(sliced_df)
    if n > 0:
        t = sliced_df.index
        s = sliced_df['Left-Vibration-Data']

        # Apply high-pass filter
        dt = (t[1] - t[0]).total_seconds()  # Sampling interval in seconds
        

        # FFT
        s -= s.mean()
        fr = np.fft.rfftfreq(n, dt)
        fou = np.fft.rfft(s)
        
        # Plot and save the FFT plot
        plt.figure(figsize=(30,8))
        plt.subplot(211)
        plt.plot(t, s,color='blue',linewidth=1)
        plt.title('Data with Noise')

        plt.subplot(212)
        plt.plot(np.fft.fftshift(fr), np.fft.fftshift(np.abs(fou)),color='blue',linewidth=1)
        plt.xticks(np.arange(min(np.fft.fftshift(fr)), max(np.fft.fftshift(fr)), 0.4))
        #plt.ylim(0, 6000)
        plt.title('Spectrum of Noisy Data')

        # Save the plot with a meaningful filename
        filename = os.path.join('D:\\1.5inch Pump Data\\With Vibration Sensor Test\\P2_8hrs data good diaphragm', f'{current_time.strftime("%H-%M-%S")}_{next_time.strftime("%H-%M-%S")}_FFT.png')
        plt.savefig(filename)
        plt.close() 
                
    else:
        pass
    current_time = next_time    
#temp_dic[key] = result_df

# Append the DataFrame to the list
Freq_Amp_df = pd.DataFrame({'Freq':Freq_list,'Amp':Amp_list})

In [ ]:
Freq_Amp_df.columns

In [67]:
from statsmodels.tsa.seasonal import seasonal_decompose

In [ ]:
plt.rcParams['figure.figsize'] = [15, 10] 
pump_add_decomposition=seasonal_decompose(Freq_Amp_df['Freq'],model='multiplicative',period=5) # Period = period of 1 wave/step = 500 ms/ 1
pump_add_decomposition.plot();

# FFT Spectrum filter

In [ ]:
Filter_data_df.head()

In [20]:

start_time = Filter_data_df.index[0]
end_time = Filter_data_df.index[-1]
time_interval = pd.Timedelta(minutes=60)

current_time = start_time
while current_time <= end_time:
    
    next_time = current_time + time_interval
    
    # Slice the DataFrame based on the specified time range
    sliced_df = Filter_data_df.iloc[(Filter_data_df.index >= current_time) & (Filter_data_df.index <= next_time)]
    
    # Perform FFT analysis only if there is data in the sliced DataFrame
    n = len(sliced_df)
    if n > 0:
        t = sliced_df.index
        s = sliced_df['Right-Vibration-Data']
        dt=(t[1]-t[0]).total_seconds()

        # FFT
        s -= s.mean()
        fr = np.fft.rfftfreq(n, dt)
        fou = np.fft.rfft(s)

        # Plot and save the FFT plot
        plt.figure(figsize=(30,15))
        plt.subplot(211)
        plt.plot(t, s,color='blue',linewidth=1,marker='.')
        plt.title('Data with Noise')

        plt.subplot(212)
        plt.plot(np.fft.fftshift(fr), np.fft.fftshift(np.abs(fou)),color='blue',linewidth=1)
        plt.xticks(np.arange(min(np.fft.fftshift(fr)), max(np.fft.fftshift(fr)), 0.3))
        #plt.ylim(0, 6000)
        plt.title('Spectrum of Noisy Data')

        # Save the plot with a meaningful filename
        filename = os.path.join('D:\\1.5inch Pump Data\\Vibration sensor with standalone power pack', f'{current_time.strftime("%Y-%m-%d %H-%M-%S")}_{next_time.strftime("%Y-%m-%d %H-%M-%S")}_FFT.png')
        plt.savefig(filename)
        plt.close()            
    else:
        pass
    current_time = next_time    

In [ ]:
xi = np.linspace(0, np.max(temp_df['Frequency']), 8000)

kde = gaussian_kde(temp_df['Frequency'], weights = temp_df['Amplitude'], bw_method = 0.005)  #tune `bw_method` to get the bandwidth you want
plt.figure(figsize=(20,5))

plt.subplot(211)
plt.plot(temp_df['Frequency'], temp_df['Amplitude'])

plt.subplot(212)
plt.plot(xi, kde.pdf(xi))

In [ ]:
Filter_data_df['Right-Vibration-Data']

In [29]:
def signaltonoise(a, axis=0, ddof=0):
    a = np.asanyarray(a)
    m = a.mean(axis)
    sd = a.std(axis=axis, ddof=ddof)
    return np.where(sd == 0, 0, m/sd)

In [ ]:
signaltonoise(Filter_data_df['Right-Vibration-Data'])

In [32]:
from scipy import stats

In [ ]:
a=stats.tstd(Filter_data_df['Right-Vibration-Data'])
a

In [ ]:
b=np.std(Filter_data_df['Right-Vibration-Data'])
b

In [ ]:
c=Filter_data_df['Right-Vibration-Data'].mean()
c

In [ ]:
SNR=c/b
SNR

# Signal.welch method

In [25]:

start_time = Filter_data_df.index[0]
end_time = Filter_data_df.index[-1]
time_interval = pd.Timedelta(minutes=5)

current_time = start_time
while current_time <= end_time:
    
    next_time = current_time + time_interval
    
    # Slice the DataFrame based on the specified time range
    sliced_df = Filter_data_df.iloc[(Filter_data_df.index >= current_time) & (Filter_data_df.index <= next_time)]
    
    # Perform FFT analysis only if there is data in the sliced DataFrame
    n = len(sliced_df)
    if n > 0:
        t = sliced_df.index
        s = sliced_df['Right-Vibration-Data'].values
        
        # FFT
        plt.figure(figsize=(10,3))
        x_freq2,y_amplitude2=welch(s,fs=30)
        plt.plot(x_freq2,y_amplitude2)

        
        # Save the plot with a meaningful filename
        filename = os.path.join('D:\\1.5inch Pump Data\\Vibration sensor with standalone power pack', f'{current_time.strftime("%Y-%m-%d %H-%M-%S")}_{next_time.strftime("%Y-%m-%d %H-%M-%S")}_FFT.png')
        plt.savefig(filename)
        plt.close()            
    else:
        pass
    current_time = next_time    

In [6]:
p=[2**i for i in range(8,13)]

In [ ]:
p

In [ ]:
f=p[2]/2.56
f

In [ ]:
5%2

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Example time-domain signal (replace this with your actual data)
# Let's assume you have a 60-second signal sampled at 33 Hz
sampling_rate = 33  # Hz
duration = 60  # seconds
t = np.linspace(0, duration, duration * sampling_rate, endpoint=False)
data = np.sin(2 * np.pi * 4 * t) + 0.5 * np.sin(2 * np.pi * 10 * t)  # Example signal with two frequencies: 4 Hz and 10 Hz

# Perform FFT
fft_output = np.fft.fft(data)
N = len(data)

# Extract real and imaginary parts
real_part = np.real(fft_output)
imaginary_part = np.imag(fft_output)

# Calculate the magnitude (amplitude)
magnitudes = np.sqrt(real_part**2 + imaginary_part**2)

# Normalize the magnitudes (optional, depending on how you want to scale your output)
normalized_magnitudes = (2.0 / N) * magnitudes

# Compute the corresponding frequencies
frequencies = np.fft.fftfreq(N, d=1/sampling_rate)

# Plot the magnitude spectrum (only the positive frequencies)
plt.plot(frequencies[:N//2], normalized_magnitudes[:N//2])
plt.xlabel('Frequency (Hz)')
plt.ylabel('Amplitude')
plt.title('Frequency Spectrum')
plt.grid(True)
plt.show()

# Find the dominant frequency
dominant_index = np.argmax(normalized_magnitudes[:N//2])
dominant_frequency = frequencies[dominant_index]
dominant_amplitude = normalized_magnitudes[dominant_index]

print(f"Dominant Frequency: {dominant_frequency} Hz")
print(f"Dominant Amplitude: {dominant_amplitude}")


In [ ]:
real_part

In [4]:
mat_set={x**2 for x in range(10)}

In [ ]:
mat_set

# RMS, Mean, Median Values

In [15]:
 
#temp_dic = {}
rms_list=[]
median_list=[]
mean_list=[]
rms_median_df = pd.DataFrame(columns=['rms','median','mean'])

start_time = Filter_data_df.index[0]
end_time = Filter_data_df.index[-1]
time_interval = pd.Timedelta(minutes=2)

current_time = start_time
while current_time <= end_time:
    
    next_time = current_time + time_interval
    
    # Slice the DataFrame based on the specified time range
    sliced_df = Filter_data_df.iloc[(Filter_data_df.index >= current_time) & (Filter_data_df.index <= next_time)]
    
    # Perform FFT analysis only if there is data in the sliced DataFrame
    n = len(sliced_df)
    if n > 0:
        #caluclate rms
        rms = np.sqrt(np.mean(sliced_df['Left-Vibration-Data']**2))
        #Median calculation
        median = sliced_df['Left-Vibration-Data'].median(numeric_only=True)
        #Mean calculation
        mean=sliced_df['Left-Vibration-Data'].mean(numeric_only=True)

        #Append coresponding maximum amplitude frequency values in to freq_amp list
        #Freq_list.append((df_Filter['Frequency']))
        #Amp_list.append((df_Filter['Amplitude']))
        rms_list.append(rms)
        median_list.append(median)  
        mean_list.append(mean)          
    else:
        pass
    current_time = next_time    
#temp_dic[key] = result_df

# Append the DataFrame to the list
rms_median_df = pd.DataFrame({'rms':rms_list,'median':median_list,'mean':mean_list})

In [ ]:
#rms_median_df=rms_median_df[rms_median_df['rms']<67.9]

In [16]:
#Reset the data frame
rms_median_df.reset_index(drop=True,inplace=True)

#Normalize the data frame values
normalized_variables = MinMaxScaler().fit_transform(rms_median_df)

#Insert the normalized vaules in to the new columns
for value_index,col_name in enumerate(rms_median_df.columns):
    rms_median_df["Norm_{}".format(col_name)] = normalized_variables[:,value_index]

#Create diaprhagm liftime
timeline = [n for n in np.linspace(0,total_hrs,len(rms_median_df))]

#Insert timeline in to dataframe
rms_median_df['Diaphragm_life'] = timeline

#Set Diaphragm life as index
rms_median_df.set_index(['Diaphragm_life'],inplace=True)

In [ ]:
fig1=plt.figure(figsize=(30,30))
a=len(rms_median_df.columns)
b=1
c=1

for i in rms_median_df.columns:
    plt.subplot(a,b,c)
    plt.plot(rms_median_df.index,rms_median_df[i])
    plt.title(i)
    plt.xticks(np.arange(min(rms_median_df.index),max(rms_median_df.index),0.5))
    plt.xlabel('Hrs')
    c=c+1

plt.show()

In [ ]:
with pd.ExcelWriter('output1.xlsx',engine='openpyxl') as writer:  
    Filter_data_df['Left-Vibration-Data'].to_excel(writer, sheet_name='Cracked_Diaphragm')
    rms_median_df['rms'].to_excel(writer, sheet_name='Cracked_rms_2min')